In [2]:
import pandas as pd

In [51]:
train_df = pd.read_parquet('../data/processed/train_split_no_fss.parquet')
test_df = pd.read_parquet('../data/processed/test_split_no_fss.parquet')

In [28]:
import re

def remove_mask_tokens(text):
    return re.sub(r"\bx+\b", " ", text.lower())

X_train = train_df["Consumer complaint narrative"].apply(remove_mask_tokens)

In [29]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

unique_words = set()

for text in X_train.dropna():
    words = re.findall(r"\b[a-z]+\b", text.lower())  # lowercase + remove ! and punctuation
    unique_words.update(w for w in words if w not in ENGLISH_STOP_WORDS)

unique_words

{'weeis',
 'cornon',
 'landdlord',
 'ratesthat',
 'foreclosing',
 'dothat',
 'airfares',
 'erronesoulsy',
 'limping',
 'righxxxx',
 'nucc',
 'tresses',
 'achopens',
 'pds',
 'holidaythat',
 'xxxxaws',
 'theheloc',
 'ignoredonmydisputes',
 'namesbnaselflndr',
 'helloi',
 'calumny',
 'illictly',
 'issuewith',
 'eill',
 'msot',
 'agemcy',
 'hammond',
 'officed',
 'miguel',
 'bully',
 'whit',
 'neutered',
 'communicationdemonstrating',
 'usaa',
 'dseveral',
 'mayxx',
 'learned',
 'chards',
 'readingroom',
 'umnaceptable',
 'matting',
 'tryng',
 'supposerd',
 'gfair',
 'miscategorizes',
 'agenciesincluding',
 'repitedly',
 'confirmationdemonstrating',
 'equifaax',
 'zapped',
 'fease',
 'usedthey',
 'atively',
 'faield',
 'supervisory',
 'thoughout',
 'pepeole',
 'collatio',
 'scheduledfor',
 'dintvrecieve',
 'newman',
 'slshas',
 'repsentative',
 'williston',
 'settlementwould',
 'emplore',
 'hadlivedhere',
 'ivy',
 'agencyo',
 'hemisphere',
 'libel',
 'paystubs',
 'tamping',
 'informationo

In [30]:
from collections import Counter

bigram_counter = Counter()

for text in X_train.dropna():
    words = [w for w in re.findall(r"\b[a-z]+\b", text.lower()) if w not in ENGLISH_STOP_WORDS]
    bigrams = zip(words, words[1:])
    bigram_counter.update(bigrams)

bigram_counter.most_common(50)

[(('credit', 'report'), 755842),
 (('u', 's'), 537396),
 (('credit', 'reporting'), 402860),
 (('s', 'c'), 369434),
 (('identity', 'theft'), 271345),
 (('fair', 'credit'), 264267),
 (('consumer', 'reporting'), 250277),
 (('reporting', 'act'), 248456),
 (('reporting', 'agency'), 243967),
 (('account', 'number'), 173278),
 (('section', 'states'), 156455),
 (('s', 'code'), 155243),
 (('credit', 'bureaus'), 150049),
 (('c', 'section'), 145712),
 (('consumer', 'report'), 145371),
 (('credit', 'card'), 125559),
 (('information', 'credit'), 113235),
 (('account', 'account'), 103935),
 (('late', 'payment'), 103482),
 (('personal', 'information'), 99412),
 (('states', 'consumer'), 99370),
 (('reporting', 'agencies'), 97976),
 (('credit', 'reports'), 89591),
 (('inaccurate', 'information'), 89370),
 (('act', 'fcra'), 89038),
 (('credit', 'file'), 87464),
 (('late', 'payments'), 85158),
 (('right', 'privacy'), 76852),
 (('agency', 'furnish'), 72939),
 (('consumer', 'credit'), 72535),
 (('date', 'o

In [14]:
bigram_counter.most_common(100)[50:100]

[(('debt', 'collection'), 52255),
 (('financial', 'protection'), 51992),
 (('payment', 'history'), 51946),
 (('violated', 'rights'), 51733),
 (('consumer', 'reports'), 51694),
 (('fraudulent', 'accounts'), 51069),
 (('wells', 'fargo'), 51066),
 (('fcra', 'u'), 50825),
 (('federal', 'law'), 50722),
 (('result', 'identity'), 50100),
 (('legal', 'action'), 49452),
 (('accounts', 'credit'), 49181),
 (('don', 't'), 48967),
 (('furnish', 'information'), 48107),
 (('following', 'items'), 47394),
 (('opened', 'balance'), 47251),
 (('information', 'relating'), 47054),
 (('federal', 'trade'), 46820),
 (('filing', 'complaint'), 46168),
 (('social', 'security'), 45954),
 (('alleged', 'debt'), 45469),
 (('trade', 'commission'), 43271),
 (('relating', 'consumer'), 43138),
 (('written', 'consent'), 42505),
 (('agency', 'person'), 42366),
 (('believe', 'information'), 42195),
 (('credit', 'bureau'), 42008),
 (('reasonable', 'cause'), 41988),
 (('protected', 'consumer'), 41961),
 (('cause', 'believe'),

In [15]:
target_bigrams = [
    ("identity", "theft"),
    ("fraudulent", "accounts"),
    ("unauthorized", "charges"),
    ("account", "locked"),
    ("cannot", "access")
]

for bg in target_bigrams:
    if bg in bigram_counter:
        print(bg, bigram_counter[bg])

('identity', 'theft') 271345
('fraudulent', 'accounts') 51069
('unauthorized', 'charges') 3188
('account', 'locked') 1917


In [16]:
keywords = {"fraud", "fraudulent", "identity", "unauthorized", "stolen", "hacked", "scam"}

candidates = [
    (bg, count)
    for bg, count in bigram_counter.items()
    if bg[0] in keywords or bg[1] in keywords
]

sorted(candidates, key=lambda x: x[1], reverse=True)[:50]

[(('identity', 'theft'), 271345),
 (('victim', 'identity'), 58790),
 (('fraudulent', 'accounts'), 51069),
 (('result', 'identity'), 50100),
 (('theft', 'fraud'), 35815),
 (('unauthorized', 'inquiries'), 24231),
 (('unauthorized', 'accounts'), 22124),
 (('ftc', 'identity'), 21199),
 (('fraudulent', 'items'), 21148),
 (('fraudulent', 'information'), 20896),
 (('fraudulent', 'account'), 19821),
 (('remove', 'fraudulent'), 14693),
 (('fraud', 'business'), 14110),
 (('fraud', 'department'), 11476),
 (('fraudulent', 'inaccurate'), 11449),
 (('resulting', 'identity'), 11311),
 (('fraudulent', 'activity'), 10939),
 (('alleged', 'identity'), 10369),
 (('fraudulent', 'inquiries'), 9706),
 (('proof', 'identity'), 9534),
 (('fraudulent', 'hard'), 9361),
 (('theft', 'identity'), 9309),
 (('fraud', 'attached'), 9169),
 (('inquiries', 'fraudulent'), 8641),
 (('inaccurate', 'unauthorized'), 8641),
 (('address', 'fraud'), 8504),
 (('considered', 'identity'), 8440),
 (('complaint', 'fraudulent'), 8430),

Priority Feature Engineering

In [41]:
# -------- IMMEDIATE --------
immediate_core = [
    "identity",
    "fraud",
    "fraudulent",
    "theft",
    "scam",
    "breach",
    "phishing",
    "impersonation",
    "hacking",
    "compromise",
    "unauthorized"
]

immediate_follow = [
    "steal","stole","stolen","stealing",
    "hack","hacked","hacking",
    "use","used","using",
    "open","opened","opening",
    "create","created","creating",
    "access","accessed","accessing",
    "charge","charged","charging",
    "withdraw","withdrew","withdrawn",
    "transfer","transferred"
]


# -------- SAME DAY --------
same_day_core = [
    "access",
    "login",
    "log in",
    "account",
    "payment",
    "transaction"
]

same_day_follow = [
    "cannot","can not","can't",
    "unable",
    "locked",
    "blocked",
    "failed",
    "declined",
    "pending",
    "stuck"
]


# -------- PRIORITY FUNCTION --------
def assign_priority(text):
    text = text.lower()

    if any(c in text for c in immediate_core) and any(v in text for v in immediate_follow):
        return "immediate"

    if any(c in text for c in same_day_core) and any(v in text for v in same_day_follow):
        return "same_day"

    return "regular"

In [33]:
X_train = X_train.to_frame()

In [42]:
X_train["priority"] = X_train['Consumer complaint narrative'].apply(assign_priority)

In [43]:
X_train.head()

,Consumer complaint narrative,priority
0,"i am aware that you have been responding to many of the disputes, complaints, and challenges with pre-written standard form letters, despite the precise information consumers have provided. i am well aware that this is illegal ; as each consumer dispute must be addressed individually and dealt with in accordance with its particular merits. my letters have been written and generated using accessible software that is available to everyone. if you choose not to process or address my letters, you are violating 15 u.s. code 5 1681i. it is against the law to delay the processing of letters from consumers based on the assumption that assistance from a third party may be available. this does not qualify as a legal exception. i am the sole author of any letters bearing my name, or they have been written by my advisors with my complete knowledge and consent. i understand that there is no requirement for you to request a power of attorney to delay processing consumer letters. if you fail to process my letters promptly, i will consider it as a deliberate disregard for my consumer rights. in such a case, i must contact my attorney and pursue legal action. \n\nheres the list of the questionable items : - account numberxxxx i don't recall having late payments on this account - account please verify and validate all data for this unproven claim including the requisite certifiable compliant reporting format standard ( s ) and processes related. please remove or prove! even document all dates and balances, personal identifiers, the 426 characters of the p6 statement, and all of its trailing fragments, and any other necessary aspect of factual reporting - account numberxxxx please remove it from my public record - account number: please supply information on how you have verified this item. \n - account numberxxxx this is not mine.",regular
1,"i am submitting this complaint myself and there is no third party involved. i attached letters to let you know more in detail about what they are reporting in my credit report. they reported inaccurate information in my credit report with inconsistent and untimely dates reported across the three bureaus, transunion, , and .",regular
2,"opened . , {$12000.00} i called this company left a message for someone to call me back they never called and still reporting. \n opened , {$160.00} i spoke with an representative and told them i dont recalled doing business with them send me a signed signature of the application they never send it. \nmedical data systems opened , {$2200.00} i spoke with an representative and told them i dont recalled doing business with them send me a signed signature of the application they never send it.",regular
3,"when this account was opened, i was a minor child at the age of . the accounts listed does not belong to me and are a result of identity theft. i have shown proof of my identity and that i was a minor child at the time the accounts were opened and the companies failed to investigate and see that i was a minor and could not have opened these accounts. they are refusing to remove collection accounts that do not belong to me. i have no knowledge of this account.",immediate
4,"i contacted each company regarding illegally pulled hard inquiry. each confirmed sent them my information and no inquiry shouldve been pulled. all inquires still remains, several are doubled on my report.",regular


In [36]:
pd.set_option("display.max_colwidth", None)

X_train[["Consumer complaint narrative", "priority"]].head(15)

,Consumer complaint narrative,priority
0,"i am aware that you have been responding to many of the disputes, complaints, and challenges with pre-written standard form letters, despite the precise information consumers have provided. i am well aware that this is illegal ; as each consumer dispute must be addressed individually and dealt with in accordance with its particular merits. my letters have been written and generated using accessible software that is available to everyone. if you choose not to process or address my letters, you are violating 15 u.s. code 5 1681i. it is against the law to delay the processing of letters from consumers based on the assumption that assistance from a third party may be available. this does not qualify as a legal exception. i am the sole author of any letters bearing my name, or they have been written by my advisors with my complete knowledge and consent. i understand that there is no requirement for you to request a power of attorney to delay processing consumer letters. if you fail to process my letters promptly, i will consider it as a deliberate disregard for my consumer rights. in such a case, i must contact my attorney and pursue legal action. \n\nheres the list of the questionable items : - account numberxxxx i don't recall having late payments on this account - account please verify and validate all data for this unproven claim including the requisite certifiable compliant reporting format standard ( s ) and processes related. please remove or prove! even document all dates and balances, personal identifiers, the 426 characters of the p6 statement, and all of its trailing fragments, and any other necessary aspect of factual reporting - account numberxxxx please remove it from my public record - account number: please supply information on how you have verified this item. \n - account numberxxxx this is not mine.",regular
1,"i am submitting this complaint myself and there is no third party involved. i attached letters to let you know more in detail about what they are reporting in my credit report. they reported inaccurate information in my credit report with inconsistent and untimely dates reported across the three bureaus, transunion, , and .",regular
2,"opened . , {$12000.00} i called this company left a message for someone to call me back they never called and still reporting. \n opened , {$160.00} i spoke with an representative and told them i dont recalled doing business with them send me a signed signature of the application they never send it. \nmedical data systems opened , {$2200.00} i spoke with an representative and told them i dont recalled doing business with them send me a signed signature of the application they never send it.",regular
3,"when this account was opened, i was a minor child at the age of . the accounts listed does not belong to me and are a result of identity theft. i have shown proof of my identity and that i was a minor child at the time the accounts were opened and the companies failed to investigate and see that i was a minor and could not have opened these accounts. they are refusing to remove collection accounts that do not belong to me. i have no knowledge of this account.",immediate
4,"i contacted each company regarding illegally pulled hard inquiry. each confirmed sent them my information and no inquiry shouldve been pulled. all inquires still remains, several are doubled on my report.",regular
5,"im asking for your assistance to investigate accounts from these companies that inaccurately report late payments. ive reviewed my credit report and found entries showing late payments where there shouldnt be anythese should not appear. please remove those remarks and correct the accounts to reflect a positive status, as previously agreed.",regular
6,"in accordance with the fair credit reporting act. the list of accounts below has violated my federally protected consumer rights to privacy and confidentiality under usc . \n\n u.s. section a. states i have

In [44]:
X_train['priority'].value_counts()

priority
regular      672001
immediate    262644
same_day     187809
Name: count, dtype: int64

In [52]:
X_test = test_df["Consumer complaint narrative"].apply(remove_mask_tokens)

In [53]:
X_test = X_test.to_frame()

In [54]:
X_test["priority"] = X_test['Consumer complaint narrative'].apply(assign_priority)

In [57]:
X_test.describe()

,Consumer complaint narrative,priority
count,280613,280613
unique,218652,3
top,"in accordance with the fair credit reporting act. the list of accounts below has violated my federally protected consumer rights to privacy and confidentiality under 15 usc 1681.\n\n15 u.s.c 1681 section 602 a. states i have the right to privacy.\n\n15 u.s.c 1681 section 604 a section 2 : it also states a consumer reporting agency can not furnish a account without my written instructions 15 u.s.c 1681c. ( a ) ( 5 ) section states : no consumer reporting agency may make any consumer report containing any of the following items of information any other adverse item of information, other than records of convictions of crimes which antedates the report by more than seven years.\n\n15 u.s.c. 1681s-2 ( a ) ( 1 ) a person shall not furnish any information relating to a consumer to any consumer reporting agency if the person knows or has reasonable cause to believe that the information is inaccurate.",regular
freq,2016,168223


In [62]:
train_df = train_df.drop(columns=["Product"])
train_df["priority"] = X_train["priority"]

In [63]:
train_df.head()

,complaint_id,Consumer complaint narrative,priority
0,1038908,"I am aware that you have been responding to many of the disputes, complaints, and challenges with pre-written standard form letters, despite the precise information consumers have provided. I am well aware that this is ILLEGAL ; as each consumer dispute must be addressed individually and dealt with in accordance with its particular merits. My letters have been written and generated using accessible software that is available to everyone. If you choose not to process or address my letters, you are violating 15 U.S. Code 5 1681i. It is against the law to delay the processing of letters from consumers based on the assumption that assistance from a third party may be available. This does not qualify as a legal exception. I am the sole author of any letters bearing my name, or they have been written by my advisors with my complete knowledge and consent. I understand that there is no requirement for you to request a Power of Attorney to delay processing consumer letters. If you fail to process my letters promptly, I will consider it as a deliberate disregard for my consumer rights. In such a case, I must contact my attorney and pursue legal action. \n\nHeres the list of the questionable items : XXXX XXXX XXXX - Account NumberXXXX I don't recall having late payments on this account XXXX XXXX XXXX - Account XXXX Please verify and validate all data for this unproven claim including the requisite certifiable XXXX XXXX Compliant Reporting Format Standard ( s ) and processes related. Please REMOVE or PROVE! Even document all dates and balances, personal identifiers, the 426 characters of the P6 statement, and all of its trailing fragments, and any other necessary aspect of factual reporting XXXX - Account NumberXXXX Please remove it from my public record XXXX - Account Number:XXXX Please supply information on how you have verified this item. \nXXXX XXXX XXXX - Account NumberXXXX This is not mine.",regular
1,341296,"I am submitting this complaint myself and there is no third party involved. I attached letters to let you know more in detail about what they are reporting in my credit report. They reported inaccurate information in my credit report with inconsistent and untimely dates reported across the three Bureaus, Transunion, XXXX, and XXXX.",regular
2,52674,"XXXX XXXX XXXX opened XXXX. XXXX, XXXX {$12000.00} I called this company left a message for someone to call me back they never called and still reporting. \nXXXX XXXX opened XXXX XXXX, XXXX {$160.00} I spoke with an representative and told them I dont recalled doing business with them send me a signed signature of the application they never send it. \nMEDICAL DATA SYSTEMS opened XXXX XXXX, XXXX {$2200.00} I spoke with an representative and told them I dont recalled doing business with them send me a signed signature of the application they never send it.",regular
3,1244099,"When this account was opened, I was a minor child at the age of XXXX XXXX XXXX. The accounts listed does not belong to me and are a result of identity theft. I have shown proof of my identity and that I was a minor child at the time the accounts were opened and the companies failed to investigate and see that I was a minor and could not have opened these accounts. They are refusing to remove collection accounts that do not belong to me. I have no knowledge of this account.",immediate
4,281,"I contacted each company regarding illegally pulled hard inquiry. Each confirmed XXXX sent them my information and no inquiry shouldve been pulled. All inquires still remains, several are doubled on my report.",regular


In [59]:
test_df = test_df.drop(columns=["Product"])
test_df["priority"] = X_test["priority"]

In [61]:
test_df.head()

,complaint_id,Consumer complaint narrative,priority
0,452450,See list of enquiries on my credit that I did not authorize and I know nothing about. \n\nXXXX XXXX XXXX XXXX XX/XX/XXXX & XX/XX/XXXX XXXX XXXX ( all banks ) XX/XX/XXXX XXXX XXXX XXXX XXXX XX/XX/XXXX,regular
1,862183,"I am unable to add my income information to my Experian Credit profile for over 90 days. I have brought this to Experian 's attention and it was assigned to Experian 's engineers and product team, according to Experian. They have no timeline of when this will be resolved nor an explanation of why it is occurring. In my opinion there is a promotional block on my credit profile internally, that they do not know how to remove.\n\nWithout being able to add my income information, it prevents me from being able to access and interpret all of the features of my credit file that resides with Experian. It also is blocking other companies from being able to view my credit file and determine eligibility for credit products. This is discrimination. XXXX other members of my very own household have absolutely XXXX problem adding their income information to their credit profiles and as a result, they are not blocked from potential creditors checking their reports for eligibility of certain credit products. \n\nI have inquired repeatedly for several months about the resolution of this and they continue to acknowledge it exists but will not resolve it or provide a timeline of when it will be resolved. Additionally I received a postcard from Experian on XX/XX/2024 telling me that Experian has removed my name from the preapproved credit offer mailing list for 5 years. I never requested this. I also am including a front and back copy of the postcard received.",same_day
2,331787,"The dispute for the account with XXXX XXXX XXXX that starts with XXXX has been received and should be completed by XXXX. \n\nSource : Experian Date Submitted : XX/XX/year> Estimated Resolution Date : XX/XX/year> Report Number : XXXX Account Number : XXXX XXXX : XXXX XXXX XXXX Address : XXXX XXXX XXXX XXXX XXXX, XXXX XXXX NV XXXX None : XXXX What now? \n\nWe'll process the disputed items with the data furnisher or vendor that collected the information from a public record. You'll receive alerts when updates are available and when your full results are ready to view. \n\nVisit dispute center",regular
3,1197086,ATM card stolen. Bank not honoring law regarding burden of proof and reporting requirements. Action will also be filed with the Office of the Comptroller of the Currency.,regular
4,1067230,"In accordance with the Fair Credit Reporting act. The List of accounts below has violated my federally protected consumer rights to privacy and confidentiality under 15 USC 1681.\n\n15 U.S.C 1681 section 602 A. States I have the right to privacy.\n\n15 U.S.C 1681 Section 604 A Section 2 : It also states a consumer reporting agency can not furnish a account without my written instructions 15 U.S.C 1681c. ( a ) ( 5 ) Section States : no consumer reporting agency may make any consumer report containing any of the following items of information Any other adverse item of information, other than records of convictions of crimes which antedates the report by more than seven years.\n\n15 U.S.C. 1681s-2 ( A ) ( 1 ) A person shall not furnish any information relating to a consumer to any consumer reporting agency if the person knows or has reasonable cause to believe that the information is inaccurate.",same_day


Save Data With Priority Column Added

In [65]:
train_df.to_parquet("../data/processed/priority_train.parquet", index=False)
test_df.to_parquet("../data/processed/priority_test.parquet", index=False)